In [1]:
def main(datasources, start_date, end_date):
    """
    滴水石穿成交量频谱因子

    计算步骤:
        1. 剔除9:30集合竞价数据及14:58后的收盘集合竞价数据
        2. 对日内分钟成交量进行IQR限幅
        3. 对成交量序列去均值并乘汉宁窗
        4. 使用rFFT计算功率谱
        5. 提取2至5分钟周期对应的频带功率
        6. 计算目标频带功率占总非直流功率的比例

    返回:
        pd.DataFrame，字段为 ['date', 'instrument', 'factor']
    """
    import numpy as np
    import pandas as pd
    import dai

    table_name = datasources["bar1m"]

    MIN_VALID_POINTS = 100
    EPSILON = 1e-12

    sql = f"""
    SELECT
        date::DATETIME AS minute_time,
        date::DATE::DATETIME AS date,
        instrument::STRING AS instrument,
        CAST(volume AS DOUBLE) AS volume
    FROM {table_name}
    WHERE date >= '{start_date}'
      AND date <= '{end_date}'
      AND volume IS NOT NULL
      AND volume >= 0
      AND NOT (
          EXTRACT(hour FROM date) = 9
          AND EXTRACT(minute FROM date) = 30
      )
      AND (
          EXTRACT(hour FROM date) < 14
          OR (
              EXTRACT(hour FROM date) = 14
              AND EXTRACT(minute FROM date) <= 57
          )
      )
    ORDER BY
        instrument,
        date,
        minute_time
    """

    minute_data = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    stk_pool = dai.query(
        """
        SELECT
            date::DATE::DATETIME AS date,
            instrument::STRING AS instrument
        FROM bigalpha_2026_instruments
        """,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    if stk_pool.empty:
        return pd.DataFrame(
            columns=["date", "instrument", "factor"]
        )

    stk_pool["date"] = pd.to_datetime(
        stk_pool["date"],
        errors="coerce",
    ).dt.normalize()

    stk_pool["instrument"] = stk_pool["instrument"].astype(str)

    stk_pool = (
        stk_pool[
            stk_pool["date"].notna()
        ]
        .drop_duplicates(
            subset=["date", "instrument"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    if minute_data.empty:
        stk_pool["factor"] = 0.0

        return (
            stk_pool[["date", "instrument", "factor"]]
            .sort_values(["date", "instrument"])
            .reset_index(drop=True)
        )

    minute_data["minute_time"] = pd.to_datetime(
        minute_data["minute_time"],
        errors="coerce",
    )

    minute_data["date"] = pd.to_datetime(
        minute_data["date"],
        errors="coerce",
    ).dt.normalize()

    minute_data["instrument"] = minute_data["instrument"].astype(str)

    minute_data["volume"] = (
        pd.to_numeric(
            minute_data["volume"],
            errors="coerce",
        )
        .replace([np.inf, -np.inf], np.nan)
    )

    minute_data = (
        minute_data[
            minute_data["minute_time"].notna()
            & minute_data["date"].notna()
            & minute_data["volume"].notna()
            & (minute_data["volume"] >= 0)
        ]
        .drop_duplicates(
            subset=["minute_time", "instrument"],
            keep="last",
        )
        .sort_values(
            ["instrument", "date", "minute_time"]
        )
        .reset_index(drop=True)
    )

    def calculate_spectral_factor(group):
        volume = group["volume"].to_numpy(
            dtype="float64",
            copy=True,
        )

        volume = volume[np.isfinite(volume)]

        if volume.size < MIN_VALID_POINTS:
            return np.nan

        q25 = np.quantile(volume, 0.25)
        q75 = np.quantile(volume, 0.75)
        midpoint = 0.5 * (q25 + q75)
        iqr = q75 - q25

        if not np.isfinite(iqr):
            return np.nan

        lower = midpoint - 3.0 * iqr
        upper = midpoint + 3.0 * iqr

        volume = np.clip(
            volume,
            lower,
            upper,
        )

        volume = volume - np.mean(volume)

        if np.std(volume) <= EPSILON:
            return 0.0

        windowed_volume = volume * np.hanning(volume.size)

        fft_coefficients = np.fft.rfft(
            windowed_volume
        )

        power_spectrum = (
            np.abs(fft_coefficients) ** 2
        )

        frequencies = np.fft.rfftfreq(
            volume.size,
            d=1.0,
        )

        valid_frequency = frequencies > 0

        target_frequency = (
            (frequencies >= 1.0 / 5.0)
            & (frequencies <= 1.0 / 2.0)
        )

        total_power = np.sum(
            power_spectrum[valid_frequency]
        )

        band_power = np.sum(
            power_spectrum[target_frequency]
        )

        if (
            not np.isfinite(total_power)
            or total_power <= EPSILON
        ):
            return 0.0

        factor_value = (
            band_power
            / (total_power + EPSILON)
        )

        if not np.isfinite(factor_value):
            return np.nan

        return float(factor_value)

    factor_data = (
        minute_data
        .groupby(
            ["date", "instrument"],
            sort=False,
        )
        .apply(calculate_spectral_factor)
        .rename("factor")
        .reset_index()
    )

    factor_data["factor"] = (
        pd.to_numeric(
            factor_data["factor"],
            errors="coerce",
        )
        .replace([np.inf, -np.inf], np.nan)
    )

    result = pd.merge(
        stk_pool,
        factor_data,
        how="left",
        on=["date", "instrument"],
    )

    result["factor"] = (
        pd.to_numeric(
            result["factor"],
            errors="coerce",
        )
        .replace([np.inf, -np.inf], np.nan)
    )

    result["factor"] = (
        result.groupby("date")["factor"]
        .transform(
            lambda x: x.fillna(x.median())
        )
        .fillna(0.0)
    )

    return (
        result[["date", "instrument", "factor"]]
        .drop_duplicates(
            subset=["date", "instrument"],
            keep="last",
        )
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )


if __name__ == "__main__":
    from bigmodule import M
    import dai

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
    }

    start_date = "2024-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    factor_data = main(
        datasources,
        start_date,
        end_date,
    )

    print("factor_data shape:", factor_data.shape)
    print(factor_data.head(10).to_string())
    print(factor_data["factor"].describe().to_string())

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=False,
        start_date=start_date,
        end_date=end_date,
    )

    print(result)

factor_data shape: (242000, 3)
        date instrument    factor
0 2024-01-02  000006.SZ  0.402913
1 2024-01-02  000012.SZ  0.595130
2 2024-01-02  000016.SZ  0.321904
3 2024-01-02  000019.SZ  0.541226
4 2024-01-02  000025.SZ  0.495706
5 2024-01-02  000028.SZ  0.272692
6 2024-01-02  000029.SZ  0.558179
7 2024-01-02  000030.SZ  0.407873
8 2024-01-02  000032.SZ  0.204022
9 2024-01-02  000034.SZ  0.244495
count    242000.000000
mean          0.414118
std           0.117100
min           0.014327
25%           0.339359
50%           0.426121
75%           0.499880
max           0.768781
[2026-07-22 17:04:47] [warning  ] bigalpha_eval._latest version='v4' (use ._latest for dev only, not for prod)
[2026-07-22 17:04:50] [info     ] bigalpha_eval.v4 开始运行 ..
[2026-07-22 17:04:51] [info     ] 对齐中证1000历史成分后，官方评估窗口: 2024-01-02 至 2024-12-31
[2026-07-22 17:04:51] [info     ] ========== 数据检查 ==========
[2026-07-22 17:04:51] [info     ] 通过：列名检查（date/instrument + 至少 1 个因子列） factor_cols=['factor', 'close